# 目的
* ~~sim_p及びsim_rによるリランキングのためのテストセット作成~~
* Jalcに登録されてある電子情報通信学会(IEICE)の和文論文誌の参考文献文字列を使用

## <font color="red">要実行</font>

In [1]:
#モジュール及び接続文字列の読み込み
import pymongo
url_mongodb = "mongodb://localhost:27017/"

#mongodbへの接続
myclient = pymongo.MongoClient(url_mongodb)
mydb = myclient["jalc"]
mycol = mydb["restapi"]

## 準備
実行は不要

In [10]:
#Jalcに登録されている電子情報通信学会の論文の数
length = mycol.count_documents({"publisher_list.publisher_name":"一般社団法人 電子情報通信学会"})
print(f"電子情報通信学会の論文：{length}件")

電子情報通信学会の論文：26800件


In [ ]:
#電子情報通信学会論文誌D システム・情報に掲載されている論文の数
#電子情報通信学会論文誌にDOIが付与され出したのがおそらく2016_10から
length = mycol.count_documents({"journal_title_name_list.journal_title_name":"電子情報通信学会論文誌D 情報・システム"})
print(f"電子情報通信学会論文誌Dの論文数：{length}")

電子情報通信学会論文誌Dの論文数：885


In [ ]:
#電子情報通信学会論文誌D_2020に掲載されている論文の数
length = mycol.count_documents({"$and":[{"journal_title_name_list.journal_title_name":"電子情報通信学会論文誌D 情報・システム"},\
                                        {"volume":"J103-D"}]})
print(f"IEICE-J-2020の論文件数：{length}")

IEICE-J-2020の論文件数：94


In [ ]:
#IEICE-J-2020-5の論文件数
length = length = mycol.count_documents({"$and":[{"journal_title_name_list.journal_title_name":"電子情報通信学会論文誌D 情報・システム"},\
                                        {"volume":"J103-D"},{"issue":"5"}]})
print(f"IEICE-J-2020-5の論文件数：{length}")

IEICE-J-2020-5の論文件数：12


## citation_listを持つものを確認
* 引用文献情報を属性として持たないものがあった

In [9]:
projection = {
    "_id":0,
    "journal":{'$arrayElemAt': ['$journal_title_name_list.journal_title_name', 0]}
}

journals = []
document_count = 0

for x in mycol.find({"$and":[{"publisher_list.publisher_name":"一般社団法人 電子情報通信学会"},\
                    {"citation_list":{"$exists":True}},{"content_language":"ja"},{"title_list.lang":"ja"}]},projection):
    document_count += 1
    if x["journal"] not in journals:
        journals.append(x["journal"])

#標準出力
for journal in journals:
    print(journal)
print(f"該当文書{document_count}件")    

Fundamentals Review
電子情報通信学会　通信ソサイエティマガジン
該当文書859件


## ***Fundamentals Review***から引用文献文字列を取得
* 電子情報通信学会 基礎・境界ソサイエティ Fundamentals Review

Fundamentals Review(2020年出版) 論文の数を確認

In [3]:
#検索クエリ
query = {
    "$and":[{"journal_title_name_list.journal_title_name":"電子情報通信学会 基礎・境界ソサイエティ Fundamentals Review"},\
            {"citation_list":{"$exists":True}},{"volume":"14"}]
}

query2 = {
    "$and":[{"journal_title_name_list.journal_title_name":"電子情報通信学会 基礎・境界ソサイエティ Fundamentals Review"},\
            {"volume":"14"},{"citation_list":{"$exists":False}}]
}
#ドキュメント数
length = mycol.count_documents(query)
length2 = mycol.count_documents(query2)
#標準出力
print(f"Fundamentals Review 14巻(2020)_引用文献文字列あり：{length}件")
print(f"Fundamentals Review 14巻(2020)_引用文献文字列なし：{length2}件")

Fundamentals Review 14巻(2020)_引用文献文字列あり：25件
Fundamentals Review 14巻(2020)_引用文献文字列なし：65件


Fundamentals Review 14巻(2020)に掲載されている論文から引用文献文字列を取得

In [28]:
#返す属性を指定
projection = {
    "_id":0,
    "citation_list":1
}

#引用文献文字列を保持するリスト
citation_text = []

#検索及び取得
for x in mycol.find(query,projection):
    for y in x["citation_list"]:
        citation_text.append(y["original_text"])

#標準出力
print(f"引用文献文字列件数：{len(citation_text)}件")

引用文献文字列件数：893件


引用文献文字列から日本語を含むものを抽出

In [31]:
import re

def has_japanese(text):
    """
    ・引用文献文字列に日本語が含まれるか否かの判定
    ・unicodeを利用
    ひらがな：u3040-u309F
    カタカナ：u30A0-u30FF
    半角カナ：uFF66-uFF9f
    一般漢字：u4E00-u9FFF
    """
    return bool(re.search(r'[\u3040-\u309F\u30A0-\u30FF\uFF66-\uFF9F\u4E00-\u9FFF]', text))

#日本語引用文献文字列
citation_jp = []

#抽出
for text in citation_text:
    if has_japanese(text):
        citation_jp.append(text)

#標準出力
print(f"日本語引用文献文字列：{len(citation_jp)}件")
for citation in citation_jp:
    print(citation)

日本語引用文献文字列：281件
(12) 谷，ルガル，“量子分散コンピューティング，” 信学論(A)，vol. J90-A, no. 5, pp. 393-402, 2007.
(4) D.E. Knuth, Surreal Numbers, Addison-Wesley, 1974, 好田順治（訳），超現実数，海鳴社，1978，至福の超現実数―純粋数学に魅せられた男と女の物語，柏書房，2004.
(5) S.W. Golomb, Polyominoes, Princeton Univ. Press, 1994. 川辺治之（訳），箱詰めパズル ポリオミノの宇宙，日本評論社，2014.
(6) M.H. Albert, R.J. Nowakowski, and D. Wolfe, Lessons in Play: An Introduction to Combinatorial Game Theory, A K Peters, 2007. 川辺治之（訳），組合せゲーム理論入門：勝利の方程式，共立出版，2011.
(7) 小林欣吾，佐藤創（監訳），数学ゲーム必勝法，共立出版，2016, 2019.
(1) 岡田　啓ほか（編），“小特集　ドローンがもたらす新しい世界，” 信学誌，vol. 101, no. 12, pp. 1161-1206, Dec. 2018.
(2) 橋口宏衛，“ドローン技術の背景，仕組み，発展，” 信学誌，vol. 101, no. 12, pp. 1162-1166, Dec. 2018.
(3) 此村　領，“人の手を介さずに飛行するロボットを実現するための要素技術と研究の興味，” 信学誌，vol. 101, no. 12, pp. 1167-1170, Dec. 2018.
(16) 磯貝海斗，猪原　亮，中野秀夫，岡崎秀晃，“学生向け組み込みハードウェアデバイスを用いたクアッドコプターの構成法，” 信学技報，vol. 115, no. 422, CAS2015-73, pp. 67-72, Jan. 2016.
(19) 磯貝海斗，“マルチロータの状態変数モデリングと飛行状態，モータ故障への応用，” 令和元年度 湘南工科大学 博士学位論文，March 2020.
(1) 貴家仁志，“技術の原点と向き合う，” IEICE Fundamentals

テキストファイルの新規作成

In [32]:
with open('ieice_reference.txt','x') as f:
    f.write('\n'.join(citation_jp))
